# Expected needs from a separate model

Keep the qualification probabilities already generated by `01-predictions_bootstrap.ipynb`
or `01-predictions_awp.ipynb`. Train **one random forest regressor per seed** to estimate
`E[gagne_sum_t | past information]`. All probability models in that seed share this need estimate.

We reproduce the original train/validation/test splits, verify the saved test-person IDs,
choose the forest's leaf size using validation RMSE, and predict needs for the test people.
The comparison baseline is simply the person's **previous-year need in its original units**.
Qualification continues to depend on actual future need; the existing `y` column is unchanged.

The output is a new probability CSV with an `expected_need` column. Actual `gagne_sum_t`
is retained for retrospective evaluation. Expected needs stay continuous: rounding belongs
in the allocation algorithm, not in this prediction notebook.

In [49]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm

# Run from either the project root or the test folder.
ROOT = Path.cwd()
if not (ROOT / "src/allocations.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src/allocations.py").exists():
    raise FileNotFoundError("Run this notebook from the project root or its test folder.")

## Settings

Choose an existing probability file. Seeds are read from that file, so there is no
separate seed count to keep synchronized. The split fractions must match the notebook
that generated it: 20% test, then 25% of the remaining 80% for validation (60/20/20 overall).

We compare three leaf sizes for a forest with 100 trees. A larger leaf averages more
training observations. Only validation RMSE chooses the winner; test outcomes do not.
The selected forest remains fitted on the training partition only.

In [50]:
PROBABILITY_METHOD = "test_bootstrap_nn"  # Or "awp".
QUALIFICATION_COLUMN = "qualified_gagne_1"
NEED_COLUMN = "gagne_sum_t"
GROUP_COLUMN = "race"
EXPECTED_NEED_COLUMN = "expected_need"
prediction_name = f"{PROBABILITY_METHOD}_{QUALIFICATION_COLUMN}"

test_size = 0.20
validation_size = 0.25
N_ESTIMATORS = 100
MIN_SAMPLES_LEAF = [1, 5, 20]
N_JOBS = -1

fp_data_raw = ROOT / "data/obermeyer/obermeyer_data_v1.csv"
fp_data_clean = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
fp_data_probs = ROOT / f"data/predictions/{prediction_name}.csv"
fp_output = ROOT / f"data/predictions/{prediction_name}_with_expected_needs.csv"
fp_metrics = ROOT / f"data/results/{prediction_name}_need_model_metrics.csv"
fp_validation = ROOT / f"data/results/{prediction_name}_need_model_validation.csv"

In [51]:
features = [
    "dem_female",
    "dem_age_band_18-24_tm1", "dem_age_band_25-34_tm1",
    "dem_age_band_35-44_tm1", "dem_age_band_45-54_tm1",
    "dem_age_band_55-64_tm1", "dem_age_band_65-74_tm1",
    "dem_age_band_75+_tm1", "hypertension_elixhauser_tm1",
    "cost_dialysis_tm1", "cost_emergency_tm1", "cost_home_health_tm1",
    "cost_ip_medical_tm1", "cost_ip_surgical_tm1", "cost_laboratory_tm1",
    "cost_op_primary_care_tm1", "cost_op_specialists_tm1", "cost_op_surgery_tm1",
    "cost_other_tm1", "cost_pharmacy_tm1", "cost_physical_therapy_tm1",
    "cost_radiology_tm1", "gagne_sum_tm1",
]

## Load raw features and existing probabilities

The cleaning notebook scales features using the whole dataset. This need model instead
uses raw features; random forests do not require scaling. Median imputation, if needed,
will be fitted inside each training split. This does not change how the existing
probability models were trained.

In the current cleaning notebook, `person_id` is the raw row index. Race is copied from
the cleaned data for group evaluation only.

In [ ]:
df_raw = pd.read_csv(fp_data_raw, usecols=features + [NEED_COLUMN, "cost_t"])
df_clean = pd.read_csv(
    fp_data_clean,
    usecols=["person_id", GROUP_COLUMN, NEED_COLUMN, "cost_t", QUALIFICATION_COLUMN],
)
df_probs = pd.read_csv(fp_data_probs)

if len(df_raw) != len(df_clean) or not np.array_equal(df_clean["person_id"], df_raw.index):
    raise ValueError("Raw and cleaned rows no longer share the original person_id mapping.")
if EXPECTED_NEED_COLUMN in df_probs.columns:
    raise ValueError("Choose the original probability file, without expected_need already added.")

data = df_raw.copy()
data["person_id"] = data.index
data[GROUP_COLUMN] = df_clean[GROUP_COLUMN].to_numpy()
data = data.set_index("person_id")
df_clean = df_clean.set_index("person_id")

seeds = sorted(df_probs["seed"].unique())
if not seeds:
    raise ValueError("The probability file contains no seeds.")
if any(not np.isfinite(seed) or seed < 0 or seed != int(seed) for seed in seeds):
    raise ValueError("Seeds must be nonnegative integers.")
print(f"Loaded {len(data):,} people and probability predictions for {len(seeds)} seeds.")

Loaded 48,784 people and probability predictions for 10 seeds.


In [60]:
splits = {}
split_sizes = []

for seed in seeds:
    train_ids, test_ids = train_test_split(data.index.to_numpy(), test_size=test_size, random_state=int(seed))
    train_ids, val_ids = train_test_split(train_ids, test_size=validation_size, random_state=int(seed))

    df_test_saved = df_probs.loc[(df_probs["seed"] == seed) & (df_probs["person_id"] >= 0)]

    # Check if the split is consistent with the split used for generating qualification probabilities
    if set(df_test_saved["person_id"]) != set(test_ids):
        raise ValueError(f"Test-person IDs do not match for seed {seed}; check the split settings.")
    saved_ids = df_test_saved["person_id"].to_numpy()
    # Ensure that the needs (gagne_sum_t) are the same
    if not np.allclose(df_test_saved[NEED_COLUMN], data.loc[saved_ids, NEED_COLUMN]):
        raise ValueError(f"Saved actual needs do not match the source data for seed {seed}.")
    # Ensure the true qualifications (gagne_sum_t >= tau) are eqaul
    if not np.array_equal(df_test_saved["y"], df_clean.loc[saved_ids, QUALIFICATION_COLUMN]):
        raise ValueError(f"Saved qualification labels do not match for seed {seed}.")
    
    splits[seed] = {"train": train_ids, "validation": val_ids, "test": test_ids}

## Fit a separate need model for each seed

Each candidate predicts continuous need and is scored on validation RMSE. We retain the
candidate with the lowest score (ties keep the first candidate). The test set is only
predicted after that choice is fixed. There is no need-model fit per probability model
or per sampled cohort.

The forest's squared-error objective estimates a conditional mean. With nonnegative
training targets its predictions are nonnegative; we neither round them nor cap them at 5.

In [59]:
validation_rows = []
evaluation_frames = []

for seed in seeds:
    train_ids = splits[seed]["train"]
    val_ids = splits[seed]["validation"]
    X_train = data.loc[train_ids, features]
    y_train = data.loc[train_ids, NEED_COLUMN]
    X_val = data.loc[val_ids, features]
    y_val = data.loc[val_ids, NEED_COLUMN]

    best_model = None
    best_rmse = np.inf
    best_leaf_size = None

    for leaf_size in MIN_SAMPLES_LEAF:
        model = make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestRegressor(
                n_estimators=N_ESTIMATORS,
                min_samples_leaf=leaf_size,
                criterion="squared_error",
                random_state=int(seed),
                n_jobs=N_JOBS,
            ),
        )
        model.fit(X_train, y_train)
        val_prediction = model.predict(X_val)
        val_rmse = root_mean_squared_error(y_val, val_prediction)
        validation_rows.append({
            "seed": seed,
            "n_estimators": N_ESTIMATORS,
            "min_samples_leaf": leaf_size,
            "validation_rmse": val_rmse,
        })

        if val_rmse < best_rmse:
            best_model = model
            best_rmse = val_rmse
            best_leaf_size = leaf_size

    if best_model is None:
        raise ValueError("MIN_SAMPLES_LEAF must contain at least one candidate.")

    for split_name in ["validation", "test"]:
        ids = splits[seed][split_name]
        frame = data.loc[ids, [NEED_COLUMN, GROUP_COLUMN, "gagne_sum_tm1"]].reset_index()
        frame[EXPECTED_NEED_COLUMN] = best_model.predict(data.loc[ids, features])
        frame["seed"] = seed
        frame["split"] = split_name
        frame["min_samples_leaf"] = best_leaf_size
        evaluation_frames.append(frame)

    print(f"Seed {seed}: min_samples_leaf={best_leaf_size}, validation RMSE={best_rmse:.3f}")

validation_results = pd.DataFrame(validation_rows)
validation_results["selected"] = False
best_rows = validation_results.groupby("seed")["validation_rmse"].idxmin()
validation_results.loc[best_rows, "selected"] = True
evaluation = pd.concat(evaluation_frames, ignore_index=True)
display(validation_results)

Seed 0: min_samples_leaf=20, validation RMSE=1.031
Seed 1: min_samples_leaf=20, validation RMSE=1.061
Seed 2: min_samples_leaf=20, validation RMSE=1.028
Seed 3: min_samples_leaf=20, validation RMSE=1.075
Seed 4: min_samples_leaf=20, validation RMSE=1.024
Seed 5: min_samples_leaf=20, validation RMSE=1.045
Seed 6: min_samples_leaf=20, validation RMSE=1.030
Seed 7: min_samples_leaf=20, validation RMSE=1.066
Seed 8: min_samples_leaf=20, validation RMSE=1.011
Seed 9: min_samples_leaf=20, validation RMSE=1.049


,seed,n_estimators,min_samples_leaf,validation_rmse,selected
0,0,100,1,1.061124,False
1,0,100,5,1.046569,False
2,0,100,20,1.030967,True
3,1,100,1,1.086575,False
4,1,100,5,1.074827,False
5,1,100,20,1.060856,True
6,2,100,1,1.062808,False
7,2,100,5,1.042384,False
8,2,100,20,1.027780,True
9,3,100,1,1.097192,False


## Check prediction errors overall and by group

Compare the selected random forest with the previous-year baseline on validation and test
data. **MAE** and **RMSE** measure prediction error. **Mean error = predicted minus actual**:
positive values indicate overprediction, negative values underprediction. Group differences
in mean error matter for the disparity calculation.

The selected model's validation performance was used for tuning. Test results are the
held-out evaluation; do not use them to choose the leaf size. These metrics describe the
full split, not the selected allocations, which need their own error audit later.

In [31]:
metric_rows = []

for (seed, split_name), split_df in evaluation.groupby(["seed", "split"]):
    for model_name, prediction_column in [
        ("random_forest", EXPECTED_NEED_COLUMN),
        ("previous_year", "gagne_sum_tm1"),
    ]:
        for group_name, group_df in [
            ("all", split_df),
            ("0", split_df.loc[split_df[GROUP_COLUMN] == 0]),
            ("1", split_df.loc[split_df[GROUP_COLUMN] == 1]),
        ]:
            if group_df.empty:
                continue
            actual = group_df[NEED_COLUMN]
            predicted = group_df[prediction_column]
            metric_rows.append({
                "seed": seed,
                "split": split_name,
                "need_model": model_name,
                "group": group_name,
                "n": len(group_df),
                "mae": mean_absolute_error(actual, predicted),
                "rmse": root_mean_squared_error(actual, predicted),
                "mean_error": (predicted - actual).mean(),
                "mean_actual_need": actual.mean(),
                "mean_predicted_need": predicted.mean(),
            })

metrics = pd.DataFrame(metric_rows)
metrics.loc[metrics["split"] == "test"]

,seed,split,need_model,group,n,mae,rmse,mean_error,mean_actual_need,mean_predicted_need
0,0,test,random_forest,all,9757,0.567596,1.039293,-0.008845,1.362919,1.354074
1,0,test,random_forest,0,8602,0.535540,0.984854,0.004099,1.259010,1.263108
2,0,test,random_forest,1,1155,0.806342,1.378696,-0.105248,2.136797,2.031548
3,0,test,previous_year,all,9757,0.440197,1.114672,0.085170,1.362919,1.448089
4,0,test,previous_year,0,8602,0.406766,1.048853,0.071262,1.259010,1.330272
5,0,test,previous_year,1,1155,0.689177,1.517574,0.188745,2.136797,2.325541
12,1,test,random_forest,all,9757,0.566337,1.025247,0.002480,1.346110,1.348591
13,1,test,random_forest,0,8617,0.542585,0.991313,0.012530,1.257398,1.269929
14,1,test,random_forest,1,1140,0.745874,1.252350,-0.073487,2.016667,1.943180
15,1,test,previous_year,all,9757,0.448089,1.111864,0.083017,1.346110,1.429128


## Attach expected needs to the original probability rows

Merge by `(seed, person_id)`, never by row position. Every real test person receives one
need estimate, shared by all their `m_...` probability columns. Original columns, order,
qualification labels, actual needs, and loss rows are preserved. Loss rows get `NaN`
for expected need because they are not individuals.

In [32]:
need_predictions = evaluation.loc[
    evaluation["split"] == "test", ["seed", "person_id", EXPECTED_NEED_COLUMN]
].copy()

output = df_probs.merge(
    need_predictions,
    on=["seed", "person_id"],
    how="left",
    sort=False,
    validate="one_to_one",
)

real_people = output["person_id"] >= 0
assert np.isfinite(output.loc[real_people, EXPECTED_NEED_COLUMN]).all()
assert (output.loc[real_people, EXPECTED_NEED_COLUMN] >= 0).all()
assert output.loc[~real_people, EXPECTED_NEED_COLUMN].isna().all()
assert output[df_probs.columns].equals(df_probs)

output.loc[real_people, ["seed", "person_id", "y", NEED_COLUMN, EXPECTED_NEED_COLUMN]].head()

,seed,person_id,y,gagne_sum_t,expected_need
0,0,2545,1.0,1.0,1.200404
1,0,8198,1.0,2.0,2.135306
2,0,46461,0.0,0.0,0.835122
3,0,30620,0.0,0.0,0.124231
4,0,47418,1.0,2.0,2.037489


In [61]:
output

,m_1,m_2,m_3,m_4,m_5,m_6,m_7,m_8,m_9,m_10,...,m_97,m_98,m_99,m_100,y,person_id,cost_t,gagne_sum_t,seed,expected_need
0,0.904931,0.871062,0.910156,0.891166,0.911215,0.856569,0.873912,0.902274,0.896905,0.894191,...,0.899107,0.880650,0.899068,0.872859,1.0,2545,5300.0,1.0,0,1.200404
1,0.915127,0.903569,0.914096,0.896051,0.886691,0.935360,0.889907,0.916145,0.897856,0.900183,...,0.895117,0.921461,0.882660,0.898648,1.0,8198,2700.0,2.0,0,2.135306
2,0.738387,0.764767,0.770606,0.763307,0.689350,0.782243,0.749848,0.780667,0.771139,0.767159,...,0.794800,0.777967,0.778475,0.758265,0.0,46461,5200.0,0.0,0,0.835122
3,0.075199,0.055435,0.058270,0.075812,0.060112,0.074473,0.041335,0.080032,0.064840,0.061019,...,0.073177,0.078605,0.061840,0.070550,0.0,30620,1300.0,0.0,0,0.124231
4,0.952070,0.944468,0.938265,0.941654,0.945965,0.953616,0.955968,0.955663,0.946939,0.950451,...,0.949779,0.949518,0.939629,0.940720,1.0,47418,2200.0,2.0,0,2.037489
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97585,0.989850,0.990080,0.985941,0.989339,0.978696,0.984411,0.981363,0.987409,0.978143,0.979244,...,0.986033,0.986533,0.992592,0.976342,1.0,28651,3500.0,3.0,9,3.270399
97586,0.076550,0.061408,0.080982,0.069586,0.061754,0.079458,0.077528,0.077342,0.070268,0.050860,...,0.075306,0.086525,0.091454,0.068576,0.0,21611,0.0,0.0,9,0.156860
97587,0.997740,0.995410,0.990378,0.990132,0.991184,0.990119,0.991838,0.993612,0.994133,0.992704,...,0.992954,0.987349,0.988149,0.991342,1.0,965,46400.0,8.0,9,6.026511
97588,0.316808,0.317442,0.317434,0.318505,0.317502,0.317089,0.316397,0.317641,0.316148,0.318201,...,0.318052,0.317632,0.317625,0.316787,NaN,-2,NaN,NaN,9,NaN


## Save

Three files are written: the augmented predictions, the error metrics, and the candidate
validation scores (including which candidate was selected). Rerunning this cell replaces
those derived files. The original probability CSV is not modified.

For a later allocation experiment, read the `_with_expected_needs.csv` file and use
`expected_need` for the allocation objective. Keep `gagne_sum_t` for actual disparity and
`y` for realized utility. Expected needs are continuous: apply the chosen integer-grid
rounding before running the integer DP. This notebook does not change `allocations.py`
or the existing experiment notebooks.

In [34]:
fp_output.parent.mkdir(parents=True, exist_ok=True)
fp_metrics.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(fp_output, index=False)
metrics.to_csv(fp_metrics, index=False)
validation_results.to_csv(fp_validation, index=False)

print(f"Saved predictions: {fp_output}")
print(f"Saved need-model metrics: {fp_metrics}")
print(f"Saved validation scores: {fp_validation}")

Saved predictions: /Users/kk985/PhD/Research/AllocationMultiplicity/data/predictions/test_bootstrap_nn_qualified_gagne_1_with_expected_needs.csv
Saved need-model metrics: /Users/kk985/PhD/Research/AllocationMultiplicity/data/results/test_bootstrap_nn_qualified_gagne_1_need_model_metrics.csv
Saved validation scores: /Users/kk985/PhD/Research/AllocationMultiplicity/data/results/test_bootstrap_nn_qualified_gagne_1_need_model_validation.csv
